In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import requests
import sqlite3

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_procs as sqlp

from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace


# Purpose and Documentation


## Purpose

This notebook 
- fetches metadata rows for image parameter(s) defined in _../config/config.yml_
- iterates over phenotyping centers and saves center-specific raw CSV files to _../data/&lt;run_name&gt;/&lt;center_data_dir&gt;_
- performs QC and merges the results into a single consolidated CSV file stored at _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/IMPC_radiographs.csv_, which is then loaded into the SQLite database _IMPC_radiographs.db_.

## Location

_&lt;project_dir&gt;/notebooks/010_fetch_data_from_impc.ipynb_

## Useful links

| # | URL  | description |
| - | :--- | :---        |
| 1 |https://www.mousephenotype.org/impress/index                                        | IMPReSS               |
| 2 |https://www.mousephenotype.org/help/programmatic-data-access/access-patterns/       | API access patterns   |
| 3 |https://www.mousephenotype.org/help/programmatic-data-access/raw-experimental-data/ | how to get raw data   |
|   |https://www.mousephenotype.org/help/programmatic-data-access/images/                | image API             |
| 4 |https://www.mousephenotype.org/help/programmatic-data-access/data-fields/           | explanation of fields |
| 5 | ---                                                                                | ---                   |
| 6 |https://api.mousephenotype.org/impress/                                             | list of APIs          |
| 7 |https://www.mousephenotype.org/phenodcc/impress-api-documentation/                  | API documentation     |
| 8 |https://www.ebi.ac.uk/mi/impc/solr/experiment/select?q=*%3A*&rows=0&facet=on&facet.field=phenotyping_center| get distinct phenotyping centers|
| 9 | --- | --- |
| 10| https://api.mousephenotype.org/impress/pipeline/list/full| all pipelines |


## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data/_ and _../reports/_ directories 
- Network access to the IMPC Solr API (<a href="https://www.ebi.ac.uk/mi/impc/solr/impc_images/select" target="_blank">https://www.ebi.ac.uk/mi/impc/solr/impc_images/select</a>)

</p>

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_file = sorted(list(Path.cwd().glob('*.ipynb')), key=lambda p: p.stat().st_ctime, reverse=True)[0].stem  

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_file}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print(" Logger successfully set up.")
except Exception as e:
    print(f" Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, config['sqlite_db'])

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlp.connect_db(sqlite_file)
    if conn is not None:
        print(f" Connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

## Download settings

In [ ]:
# this is the base URL for the IMPC raw data API
api_base_url = "https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:"

# example: get one row of data
# https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:IMPC_XRY_034_001%20AND%20colony_id:MEAW&wt=csv&rows=1

# here we define which filters we want to apply
api_filters = ("")

# here we define which fields we want to have
api_fields  = ("&fl=phenotyping_center,pipeline_name,pipeline_stable_id,life_stage_name,external_sample_id,date_of_birth,"
               "gene_symbol,sex,biological_sample_group,strain_name,procedure_name,procedure_stable_id,age_in_days,"
               "age_in_weeks,parameter_stable_id,parameter_name,litter_id,gene_accession_id,zygosity,file_type,"
               "omero_id,date_of_experiment")

# here we define additional options
api_options  = '&start=0&wt=csv'

# mode: REAL=download everything, TEST=only 20 rows
mode = "TEST"

# Fetch metadata from IMPC

## Expected parameter values by center

In [ ]:
my_parameters = pd.DataFrame(config['parameters'])
my_parameters

In [ ]:
# initialise dictionary
centers_numbers = defaultdict(lambda: defaultdict(int))

total_number = 0

# loop over all parameters
for my_stable_id in my_parameters.stable_id:
    
    total_number = 0
    
    my_url = f"https://www.ebi.ac.uk/mi/impc/solr/impc_images/select?q=parameter_stable_id:{my_stable_id}&rows=0&facet=on&facet.field=phenotyping_center"
    
    response = requests.get(my_url)

    if response.status_code == 200:
        data = response.json()

        numbers = data["facet_counts"]["facet_fields"]["phenotyping_center"]

        # build dictionary centers_numbers[center][parameter]
        for i in range(0, len(numbers), 2):
            centers_numbers[numbers[i]][my_stable_id] = numbers[i+1]
            total_number += numbers[i+1]

# convert dictionary to dataframe
expected_df = pd.DataFrame(centers_numbers)
print(f"total number of whole body dorso-ventral radiographs: {total_number}")

# show the 
expected_df

## Data per center and parameter

In [ ]:
my_centers = config['centers']
my_centers

In [ ]:
number_centers = len(my_centers)
for my_stable_id in my_parameters.stable_id:
 
    logger.info(f"\n==================================================================================================================================")
    logger.info(f"DOWNLOADING: {my_stable_id} from {str(number_centers)} centers") 
    logger.info(f"==================================================================================================================================")

    counter = 0
    
    # loop over all phenotyping centers
    for my_center in my_centers:
        
        if my_center not in expected_df.columns:
            continue

        counter = counter + 1

        # build file name
        my_center_stripped = my_center.replace(" ", "_")
        my_file = os.path.join(dirs.raw_data_dir, f"parameter-{my_stable_id}-{my_center_stripped}-raw_data.csv")
        # build center filter
        my_center_quoted   = '\"' + my_center + '\"'
        my_center_filter = "&fq=phenotyping_center:" + my_center_quoted

        # get expected number of data points
        expected_data_points = expected_df[my_center][my_stable_id]

        if mode=="TEST":
            # in TEST mode, only download 20 rows of data
            api_lines = '&rows=20'
        else:
            # dynamically set number of rows to download (expected + 100)
            api_lines  = '&rows=' + str(expected_data_points + 10)

        # build the URL from the components defined above
        my_url = api_base_url + my_stable_id + my_center_filter + api_fields + api_lines + api_options
        
        logger.setLevel(logging.INFO)
        logger.info(f"\n{str(counter)}. trying to download expected {expected_data_points} data points of {my_stable_id} from {my_center} using:\n{my_url}")

        # replace whitespaces, e.g. in "MRC Harwell" and "UC Davis"
        my_url_quoted = my_url.replace(" ", "%20")
        
        print(my_url)
        response = requests.get(my_url)
        response.raise_for_status()  # Fehler werfen, falls HTTP != 200

        outfile = Path(my_file)
        print(outfile)
        outfile.write_bytes(response.content)

        # os.system(command_string)
        with open(my_file) as f:
             row_count = sum(1 for line in f)

        # ERROR: less data than expected
        if row_count < expected_data_points: 
           logger.error(f"\n############ WARNING: {expected_data_points} expected, but only {row_count - 1} downloaded! ############")

        # error or not: inform
        logger.info(f"\ndownloaded {row_count - 1} rows of data and stored in: {my_file}\n-----------\n")
    
    logger.info(f"\n==================================================================================================================================")

# Iterate centers, run QC and combine outputs into one file

In [ ]:
my_max_pipeline_deviation = config['max_pipeline_deviation']
my_mouse_columns = config['mouse_columns']

In [ ]:
number_centers = len(my_centers)
number_params  = len(my_parameters)
center_counter = 0
param_counter  = 0

# list to collect all center dataframes
all_df_list = []

print(f"reading raw data from: {dirs.raw_data_dir} ")

# loop over all phenotyping centers
for my_center in my_centers:
    my_center_stripped = my_center.replace(" ", "_")

    center_counter = center_counter + 1

    print(f"\n==================================================================================================================================")
    print(f"Center {center_counter} of {number_centers}: {my_center}")
    logger.info(f"\n==================================================================================================================================")
    logger.info(f"PROCESSING CENTER {center_counter} OF {number_centers}: {my_center} - checking for {number_params} parameter files") 
    logger.info(f"==================================================================================================================================")

    # dataframe to collect data from all parameters of current center
    combined_df = pd.DataFrame()

    # loop over all parameters
    for idx in my_parameters.index:
        parameter_id = my_parameters.at[idx, "stable_id"]
        description  = my_parameters.at[idx, "description"]
        due_week     = int(my_parameters.at[idx, "week"])
        work_name    = my_parameters.at[idx, "work_name"]

        param_counter = param_counter + 1

        print(f"   Parameter {param_counter} of {number_params}: {parameter_id} ({work_name})")
        logger.info(f"Parameter {param_counter} of {number_params}: {parameter_id} ({work_name}) ")
        
        # build expected file name for raw data of current center and current parameter
        my_file = os.path.join(dirs.raw_data_dir, f"parameter-{parameter_id}-{my_center_stripped}-raw_data.csv") 

        # check if expected file exists: if yes - process it
        if os.path.exists(my_file):
           
            logger.info(f"processing \"./{my_file}\"... ")
            df = pd.read_csv(my_file) 

            # inform
            logger.info(f"   {str(len(df))} rows in file")
            logger.info(f"   distinct values in columns, sorted - age week should be {str(due_week)}")
            for col in ["sex", "age_in_weeks", "pipeline_stable_id", "life_stage_name", "biological_sample_group", "strain_name"]:
                unique_vals = df[col].dropna().unique()
                unique_vals_str = sorted([str(x) for x in unique_vals])
                logger.info(f"      {col}: {unique_vals_str}")
            logger.info(f"   distinct entries")
            
            for col in ["external_sample_id", "gene_symbol"]:
                logger.info(f"      {col}: {len(df[col].unique())}")

            #----------------------------------------------------
            # duplicates filter
            duplicates = df[df.duplicated(subset=['external_sample_id', 'omero_id'], keep=False)]  # get duplicates
            logger.info(f"   duplicates (same mouse & same value) removed: {str(len(duplicates.external_sample_id.unique()))}")
            df = df.drop_duplicates(subset=['external_sample_id', 'omero_id'], keep='first')       # remove them, keep first
            #logger.info(str(len(duplicates.external_sample_id.unique())))                         # inform

            # handle age week
            weeks = df["age_in_weeks"].value_counts().to_string()
            weeks_str = str(weeks)
            logger.info(f"   " + weeks_str.replace('\n', '\n      '))

            #----------------------------------------------------
            # pipeline filter: remove all entries that deviate from the due_week according to the pipeline
            initial_length = len(df)
            df_filtered = df[(df['age_in_weeks'] >= due_week - my_max_pipeline_deviation) & 
                             (df['age_in_weeks'] <= due_week + my_max_pipeline_deviation)
            ]

            removed_rows = initial_length - len(df_filtered)

            # apply changes and report
            df = df_filtered

            if removed_rows > 0:
               print(f"       removed {removed_rows} mice, since measure week deviated more than {my_max_pipeline_deviation} weeks from due week given by pipeline!")
               logger.info(f"      removed {removed_rows} mice, since measure week deviated more than {my_max_pipeline_deviation} weeks from due week given by pipeline!")
            #----------------------------------------------------
            
            # rebuild dataframe
            # 1) drop columns
            df = df.drop(columns=['pipeline_name', 'pipeline_stable_id','life_stage_name','procedure_name', 'procedure_stable_id'])
            df = df.drop(columns=['age_in_days', 'parameter_stable_id', 'parameter_name', 'age_in_weeks'])
            
            # 2) rename columns
            df = df.rename(columns={'phenotyping_center'      : 'center',
                                    'date_of_birth'           : 'dob',
                                    'biological_sample_group' : 'cohort_type',
                                    'strain_name'             : 'strain', 
                                    'file_type'               : 'impc_file_type'
                                })
            df["mouse_id"] = df["center"].astype(str) + "_" + df["external_sample_id"].astype(str)
            
            # 3) sort columns
            df = df[list(my_mouse_columns) + ['impc_file_type', 'omero_id', 'date_of_experiment']]

            # 4) change date format
            df['dob'] = pd.to_datetime(df['dob']).dt.strftime('%Y-%m-%d')
            
            # add new columns
            df['downloaded'] = 'no'
            
            # change the values in cohort_type
            df['cohort_type'] = df['cohort_type'].replace({'experimental': 'mutant'})

            #----------------------------------------------------
            # now we remove all entries where date of birth (dob) before "2010-11-15"
            initial_length = len(df)
            df_filtered = df[(df['dob'] >= "2010-11-15")]

            removed_rows = initial_length - len(df_filtered)

            # apply changes and report
            df = df_filtered

            if removed_rows > 0:
               print(f"       removed {removed_rows} mice born before 2011 (non-IMPC mice)")
               logger.info(f"      removed {removed_rows} mice born before 2011 (non-IMPC mice)")
            #---------------
            
            # merge df to combined_df to create a center-specific combined dataframe
            # merge is based on key_columns
            key_columns = my_mouse_columns
            
            if combined_df.empty:
               combined_df = df
            else:
               logger.info(f"   merging data ...")
               combined_df = pd.merge(combined_df, df, on=key_columns, how='outer')
        
        # in case no file for current parameter found
        else:
            print(f"   file {my_file} NOT FOUND!")
            logger.info(f"file {my_file} ===========> NOT FOUND! ")
        print(" ")

        # done with current parameter

    # done with current center
  
    logger.info(f"total rows (after filter): {len(combined_df)}")

    # ------------------------------------------------    

    # add center dataframe to list for overall IMPC dataframe
    all_df_list.append(combined_df)
    
    # save combined_df to file
    out_filename = os.path.join(dirs.center_data_dir, f"IMPC-{my_center_stripped}-data.csv")
    print(f"saving combined data for center \"{my_center}\" to {out_filename}")
    logger.info(f"\nsaving combined data for center \"{my_center}\" to {out_filename}")
    combined_df.to_csv(out_filename, index=False)

    # reset parameter counter, be ready for next center
    param_counter = 0     

# done with all centers, now create overall IMPC dataframe
all_df = pd.concat(all_df_list, ignore_index=True)        # concat all center dataframes
all_df['mouse_id'] = all_df['mouse_id'].astype(str)       # convert mouse_ids to string
all_df = all_df.sort_values(by='mouse_id')                # sort by mouse_id
all_df = all_df.reset_index(drop=True)                    # re-index

# create logger
logger = logging.getLogger('all')
logger.setLevel(logging.INFO)

# save overall IMPC media files data csv file
out_filename = os.path.join(dirs.all_data_dir, "IMPC_radiographs.csv")
print(f"\n----------\nsaving combined data for all centers to {out_filename}")
logger.info(f"\nsaving combined data for all centers {out_filename}")
all_df.to_csv(out_filename, index=False)

logger.info(f"Total: {len(all_df)-1} mice")
logger.info(f"Total: {all_df['gene_symbol'].nunique()} genes")

# copy dataframe (we need it later to merge it with weights dataframe
all_data_df = all_df.copy()

In [ ]:
print(f" {all_data_df.mouse_id.nunique()} \tdistinct mouse IDs")
print(f" {all_data_df.omero_id.nunique()}\tdistinct OMERO IDs")
print(f" {all_data_df['gene_symbol'].nunique()} \tdistinct genes")

## Write DataFrame to SQLite database

In [ ]:
all_data_df

In [ ]:
conn = sqlite3.connect(sqlite_file)
all_data_df.to_sql("impc_data", conn, if_exists="replace", index=False)
conn.close()